# Dependencies

In [ ]:
!pip install notifiers

# Token


In [ ]:
token = ''

chatId = []


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import pipeline

tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert")

In [ ]:
import yfinance as yf

assets = {
    "^GSPC": [],
    "SPX": [],
    "SPY": [],
    "^MID": [],
    "IJH": [],
    "MDY": [],
    "NDAQ": [],
    "NDX": [],
    "QQQ": []
}

periodity_sec = 60


In [ ]:
def get_sentiment(text, tokenizer, model):
    classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
    result = classifier(text)
    sentiment_class = result[0].get('label', 'Not defined')
    sentiment_score = result[0].get('score', 'Not defined')
    print(sentiment_class)
    print(sentiment_score)
    return sentiment_class, sentiment_score

In [ ]:
def extract_content(content):
    title = content.get("title", "")
    description = content.get("description", "")
    summary = content.get("summary", "")
    prompt = f"{title}\n{description}\n{summary}"
    return title, prompt


In [ ]:
import html
import datetime

def format_news_for_telegram(news_item, sentiment_class, sentiment_score):
    def escape_md(text):
        return text

    def escape_html(text):
        return html.escape(text)

    def sentiment_emoji(label):
        return {
            "positive": ("Bullish", "🟢📈"),
            "neutral": ("Neutral", "🟡"),
            "negative": ("Bearish", "🔴📉")
            }.get(label, "Neutral ⚪")

    print(news_item)

    tickers = ', '.join(news_item.get("relatedTickers", []))
    title = news_item.get("title", "Без назви")
    sentiment = sentiment_emoji(sentiment_class)
    score = float(sentiment_score) * 100
    source = news_item.get("publisher", "Невідомо")
    url = news_item.get("link", "#")
    dt = datetime.utcfromtimestamp(news_item.get('providerPublishTime', 0))
    formatted_time = dt.strftime('%Y-%m-%d %H:%M UTC')

    text = (
        f"📌 *Тікери:* {escape_md(tickers)}\n"
        f"📰 *Назва:* {escape_md(title)}\n"
        f"🕒 *Опубліковано:* {formatted_time}\n"
        f"📊 *Сентимент:* {sentiment[1]} {escape_md(sentiment[0])}\n"
        f"📈 *Оцінка:* {score:.1f}\\%\n"
        f"📝 *Джерело:* {escape_md(source)}\n"
        f"🔗 [Читати далі]({escape_md(url)})"
    )

    return text

In [ ]:
def check_news(assets):
    for k,v in assets.items():
        news = yf.Search(k, news_count=3).news
        cur_news = []
        for new_item in news:
            print(new_item)
            cur_news.append(new_item["uuid"])
            if new_item["uuid"] not in assets[k]:
                title, prompt = extract_content(new_item)
                sentiment_class, sentiment_score = get_sentiment(prompt, tokenizer, model)
                text = format_news_for_telegram(new_item, sentiment_class, sentiment_score)

                print(text)
                yield text
        assets[k] = cur_news


# Messages

In [ ]:
from notifiers import get_notifier
import time
from datetime import datetime

telegram = get_notifier('telegram')
while True:
    now_utc = datetime.utcnow()
    print(now_utc.strftime('%Y-%m-%d %H:%M:%S UTC'))
    for mes in check_news(assets):
        telegram.notify(token=token, chat_id=chatId[0], message=mes, parse_mode='markdown')
        telegram.notify(token=token, chat_id=chatId[1], message=mes, parse_mode='markdown')
    time.sleep(periodity_sec)

